**Regresion Logistica**

In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from skimage.feature import hog

# Ruta a la carpeta principal que contiene las subcarpetas
ruta_carpeta_general = 'train/'

# Listas para almacenar las imágenes y etiquetas
imagenes = []
etiquetas = []

# Función para extraer características HOG
def extraer_caracteristicas_hog(imagen):
    return hog(imagen, pixels_per_cell=(16, 16), cells_per_block=(2, 2), orientations=9)

# Función para extraer características Fourier
def extraer_caracteristicas_fourier(imagen):
    f_transformada = np.fft.fft2(imagen)
    f_shifted = np.fft.fftshift(f_transformada)  
    magnitud_espectro = np.abs(f_shifted)        
    # Redimensionamos para reducir tamaño y convertimos en vector
    magnitud_espectro = cv2.resize(magnitud_espectro, (32, 32)).flatten()
    return magnitud_espectro

# Recorrer las subcarpetas
for subcarpeta in os.listdir(ruta_carpeta_general):
    ruta_subcarpeta = os.path.join(ruta_carpeta_general, subcarpeta)
    if os.path.isdir(ruta_subcarpeta):
        for archivo in os.listdir(ruta_subcarpeta):
            ruta_imagen = os.path.join(ruta_subcarpeta, archivo)
            # Cargar la imagen en escala de grises y redimensionarla
            imagen = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
            imagen = cv2.resize(imagen, (128, 128))
            # Extraer características HOG y Fourier
            caracteristicas_hog = extraer_caracteristicas_hog(imagen)
            caracteristicas_fourier = extraer_caracteristicas_fourier(imagen)
            # Combinar características de HOG y Fourier
            caracteristicas_combinadas = np.concatenate([caracteristicas_hog, caracteristicas_fourier])
            imagenes.append(caracteristicas_combinadas)
            etiquetas.append(subcarpeta)

# Convertir listas a arrays de NumPy
X = np.array(imagenes)
y = np.array(etiquetas)

# Escalado de características antes de PCA
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Aplicar PCA para reducir la dimensionalidad
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X)

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.1, random_state=42)

# Crear y entrenar el modelo de regresión logística
modelo = LogisticRegression(max_iter=2000, solver='liblinear', C=0.1)
modelo.fit(X_train, y_train)

# Evaluar el modelo
y_pred = modelo.predict(X_test)
precision = accuracy_score(y_test, y_pred)
print(f'Precisión del modelo: {precision * 100:.2f}%')

Precisión del modelo: 40.30%


In [2]:
import pandas as pd

# Ruta a la carpeta de prueba (test)
ruta_carpeta_test = 'test/'

# Listas para almacenar las imágenes y nombres de archivo
imagenes_test = []
nombres_archivos = []

# Función para extraer características HOG (misma que en el entrenamiento)
def extraer_caracteristicas_hog(imagen):
    return hog(imagen, pixels_per_cell=(16, 16), cells_per_block=(2, 2), orientations=9)

# Función para extraer características Fourier (misma que en el entrenamiento)
def extraer_caracteristicas_fourier(imagen):
    f_transformada = np.fft.fft2(imagen)
    f_shifted = np.fft.fftshift(f_transformada)
    magnitud_espectro = np.abs(f_shifted)
    magnitud_espectro = cv2.resize(magnitud_espectro, (32, 32)).flatten()
    return magnitud_espectro

# Recorrer la carpeta de prueba y ordenar los archivos alfabéticamente
for archivo in sorted(os.listdir(ruta_carpeta_test)):
    ruta_imagen = os.path.join(ruta_carpeta_test, archivo)
    # Cargar la imagen en escala de grises y redimensionarla al mismo tamaño que las imágenes de entrenamiento
    imagen = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    if imagen is not None:
        imagen = cv2.resize(imagen, (128, 128))  
        # Extraer características HOG y Fourier
        imagen_hog = extraer_caracteristicas_hog(imagen)
        imagen_fourier = extraer_caracteristicas_fourier(imagen)
        # Combinar características de HOG y Fourier
        imagen_caracteristicas = np.concatenate([imagen_hog, imagen_fourier])
        imagenes_test.append(imagen_caracteristicas)
        nombres_archivos.append(archivo)

# Convertir la lista de características combinadas a un array de NumPy
X_test = np.array(imagenes_test)

# Escalar las características usando el mismo escalador del entrenamiento
X_test = scaler.transform(X_test)

# Reducir dimensiones con PCA
X_test_pca = pca.transform(X_test)

# Hacer predicciones con el modelo entrenado
predicciones = modelo.predict(X_test_pca)

# Crear un DataFrame para almacenar los resultados
resultados = pd.DataFrame({
    'file': nombres_archivos,   
    'label': predicciones       
})

# Guardar los resultados en un archivo CSV
resultados.to_csv('Reglog.csv', index=False)

**Red Neuronal Convolucional**

In [3]:
import glob
import cv2
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input, BatchNormalization, GlobalAveragePooling2D, Rescaling, RandomZoom, RandomRotation, RandomFlip, Rescaling
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data import AUTOTUNE
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
imagenes_train = []
labels_train = []

for label, folder in enumerate(glob.glob("train/*")):
    for img_path in glob.glob(f"{folder}/*.png"):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        img = cv2.resize(img, (128, 128)).flatten()
        imagenes_train.append(img)
        labels_train.append(label)
    
imagenes_train = np.array(imagenes_train)
labels_train = np.array(labels_train)

In [5]:
batch_size = 32
img_height = 180
img_width = 180
dir = "train"

train_dataset = image_dataset_from_directory(
  dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

validacion_dataset = image_dataset_from_directory(
  dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

image_class_labels = train_dataset.class_names

Found 5406 files belonging to 16 classes.
Using 4325 files for training.
Found 5406 files belonging to 16 classes.
Using 1081 files for validation.


In [6]:
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validacion_dataset = validacion_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [7]:
num_classes = len(image_class_labels)

data_augmentation = Sequential([
  RandomFlip("horizontal", input_shape=(img_height, img_width, 3)),
  RandomRotation(0.1),
  RandomZoom(0.1),
])

model = Sequential([
  data_augmentation,
  Rescaling(1./255, input_shape=(img_height, img_width, 3)),
  Conv2D(16, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Conv2D(32, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Conv2D(64, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Flatten(),
  Dense(128, activation='relu'),
  Dense(num_classes, activation='softmax')
])

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
model.compile(optimizer='adam',loss=SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])

In [11]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(
  train_dataset,
  validation_data=validacion_dataset,
  epochs=50,
  callbacks=[early_stopping]
)

validacion_predicciones = model.predict(validacion_dataset)
validacion_predicciones_clases = np.argmax(validacion_predicciones, axis=1)
val_labels = np.concatenate([y for x, y in validacion_dataset], axis=0)

print("Matriz de confusión:\n", confusion_matrix(val_labels, validacion_predicciones_clases))

Epoch 1/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 22s 160ms/step - accuracy: 0.8097 - loss: 0.5705 - val_accuracy: 0.7641 - val_loss: 0.7277
Epoch 2/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 22s 160ms/step - accuracy: 0.8404 - loss: 0.4655 - val_accuracy: 0.7586 - val_loss: 0.8226
Epoch 3/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 23s 166ms/step - accuracy: 0.8290 - loss: 0.4793 - val_accuracy: 0.7391 - val_loss: 0.8837
Epoch 4/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 21s 157ms/step - accuracy: 0.8483 - loss: 0.4435 - val_accuracy: 0.8196 - val_loss: 0.5815
Epoch 5/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 22s 163ms/step - accuracy: 0.8599 - loss: 0.4265 - val_accuracy: 0.7965 - val_loss: 0.6651
Epoch 6/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 22s 165ms/step - accuracy: 0.8503 - loss: 0.4206 - val_accuracy: 0.7983 - val_loss: 0.6804
Epoch 7/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 23s 166ms/step - accuracy: 0.8833 - loss: 0.3410 - val_accuracy: 0.7521 - val_loss: 1.0024
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step
Matriz de confusión Regresion Logistica:
 [

2025-05-23 11:05:06.333655: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [12]:
dir_test = "test"
ruta_img_test = glob.glob(os.path.join(dir_test, "*.png"))

def load_and_preprocess_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_height, img_width])
    img = tf.expand_dims(img, 0)
    return img

predicciones = []
nombres_imagenes = []

for img_path in ruta_img_test:
        img = load_and_preprocess_image(img_path)
        preds = model.predict(img)
        predicciones_class = np.argmax(preds, axis=1)
        predicted_class_name = image_class_labels[predicciones_class[0]]
        predicciones.append(predicted_class_name)
        nombres_imagenes.append(img_path)

resultados = pd.DataFrame({
        "file": [os.path.basename(name) for name in nombres_imagenes],
        "label": predicciones
})

resultados.to_csv("Resultado_cnn.csv", index=False)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━